# Nuage de points sur carte : éditions de Venise, Paris et Lyon

Pour ces trois villes d'édition des *Métamorphoses* d'Ovide, un nuage de points sur une
**vraie carte géographique historique** : Venise, Paris et Lyon sont à leur
position réelle, et autour de chaque ville un petit nuage de points en spirale, un point par
**éditeur** ayant publié dans cette ville (taille ∝ nombre d'éditions publiées). Avec un tableau
dépliable liste tout pour chaque ville.

Même source de données que `01_carte_circulation.ipynb` : `retours_celine/BNU_corpus.ods`
(feuille `Synthèse`).

In [1]:
import os
import re
import json
from odf.opendocument import load as charger_ods
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf import teletype

RACINE = os.path.abspath("../..")
DOSSIER_VIZ = os.path.join(RACINE, "resultats", "Datavis")
os.makedirs(DOSSIER_VIZ, exist_ok=True)
CHEMIN_SORTIE = os.path.join(DOSSIER_VIZ, "nuage_editions_venise_paris_lyon.html")
CHEMIN_CORPUS = os.path.join(RACINE, "retours_celine", "BNU_corpus.ods")

## 1. Chargement du corpus

Même lecteur ODS cellule par cellule que pour la carte (`pandas.read_excel` masque des
colonnes de ce fichier — voir `01_carte_circulation.ipynb` pour le détail). On ne garde que les
éditions dont la ville normalisée est Lyon, Paris ou Venise.

In [2]:
def lire_feuille_ods(chemin, nom_feuille):
    """Lit une feuille ODS cellule par cellule (pandas ignore des colonnes de ce fichier)."""
    doc = charger_ods(chemin)
    table = next(t for t in doc.spreadsheet.getElementsByType(Table)
                 if t.getAttribute("name") == nom_feuille)
    lignes_brutes = table.getElementsByType(TableRow)

    def valeurs_ligne(ligne):
        valeurs, col = {}, 0
        for cellule in ligne.getElementsByType(TableCell):
            rep = cellule.getAttribute("numbercolumnsrepeated")
            rep = int(rep) if rep else 1
            paras = cellule.getElementsByType(P)
            texte = " ".join(teletype.extractText(p) for p in paras)
            for k in range(rep):
                valeurs[col + k] = texte
            col += rep
        return valeurs

    entetes = valeurs_ligne(lignes_brutes[0])
    colonnes = {i: t.strip() for i, t in entetes.items() if t.strip()}

    lignes = []
    for ligne in lignes_brutes[1:]:
        rep = ligne.getAttribute("numberrowsrepeated")
        rep = int(rep) if rep else 1
        valeurs = valeurs_ligne(ligne)
        if not any(v.strip() for v in valeurs.values()):
            continue  # ligne vide (fin de feuille)
        d = {nom: valeurs.get(i, "").strip() for i, nom in colonnes.items()}
        lignes.extend([d] * rep)
    return lignes

def extraire_annee(valeur):
    """Renvoie la première année à 4 chiffres trouvée (ex: "1527 / 1528 ?" -> 1527)."""
    m = re.search(r"\d{4}", str(valeur))
    return int(m.group()) if m else None

def extraire_lien(row):
    """Choisit le premier lien exploitable, par ordre de préférence (même logique que
    01_carte_circulation.ipynb) : certaines cellules contiennent plusieurs liens séparés par
    ';' (ex. plusieurs notices catalogue) — on ne garde que le premier, sinon le lien final
    serait une concaténation invalide (ex. Lyon 1516, Biblioteca Digital Ovidiana contenait
    une seule URL mais suivie d'un ';?%3E' résiduel qui aurait été inclus tel quel)."""
    for col in ["version numérisée 1", "version numérisée 2", "Biblioteca Digital Ovidiana", "url catalogue"]:
        val = row.get(col, "")
        if not val:
            continue
        premier = val.split(";")[0].strip()
        if premier.startswith("http"):
            return premier
    return None

def graveur_ou_inconnu(g):
    """Repli neutre pour un graveur non identifié ('?', 'inaccessible', case vide) — les
    'AnonymeXXXX' (X = année) sont eux déjà des identifiants distincts d'un graveur à l'autre,
    contrairement à 's.n.' pour les éditeurs (voir plus bas) : pas besoin de les numéroter."""
    g = (g or "").strip()
    if not g or g.lower() in {"?", "inaccessible"}:
        return "Graveur non identifié"
    return g

# Seules les 3 villes qui nous intéressent ici 
VILLES_RETENUES = {"Paris": "Paris", "[Paris]": "Paris", "Lyon": "Lyon", "Venise": "Venise"}
ORDRE_VILLES = ["Lyon", "Paris", "Venise"]

COL_GRAVEUR = "graveur\xa0: Nom, Prénom"

corpus = lire_feuille_ods(CHEMIN_CORPUS, "Synthèse")

def fusionner_tomes(corpus):
    """Une même édition est parfois scindée en plusieurs tomes dans le tableau source (une
    ligne par tome : même ville/année/éditeur/titre abrégé/graveur, seul le titre complet
    varie selon le tome — ex. Lyon 1697 "Les Oeuvres d'Ovide..." en 3 tomes, Amsterdam 1693 en
    3 tomes). On les fusionne en une seule édition : sinon elles compteraient 2 ou 3 fois pour
    ce qui est en réalité une seule publication. Le lien et le commentaire de copie, souvent
    renseignés sur un seul des tomes, sont récupérés du premier tome qui les a."""
    groupes, ordre = {}, []
    for ligne in corpus:
        cle = (ligne.get("ville", ""), ligne.get("année", ""), ligne.get("publisher", ""),
               ligne.get("titre abrégé", ""), ligne.get(COL_GRAVEUR, ""))
        if cle not in groupes:
            groupes[cle] = []
            ordre.append(cle)
        groupes[cle].append(ligne)

    champs_premier_non_vide = ["url catalogue", "version numérisée 1", "version numérisée 2",
                                "Biblioteca Digital Ovidiana", "copies de cette édition"]
    fusionne = []
    for cle in ordre:
        lignes_tomes = groupes[cle]
        base = dict(lignes_tomes[0])
        if len(lignes_tomes) > 1:
            for champ in champs_premier_non_vide:
                for ligne in lignes_tomes:
                    if ligne.get(champ, "").strip():
                        base[champ] = ligne[champ]
                        break
        fusionne.append(base)
    return fusionne

nb_avant_fusion = len(corpus)
corpus = fusionner_tomes(corpus)
if len(corpus) != nb_avant_fusion:
    print(nb_avant_fusion - len(corpus), "lignes fusionnées (tomes d'une même édition regroupés)")

editions = []
for row in corpus:
    ville = VILLES_RETENUES.get(row.get("ville", "").strip())
    if ville is None:
        continue
    annee = extraire_annee(row.get("année", ""))
    if annee is None:
        continue
    titre = row.get("titre abrégé") or row.get("titre complet") or ""
    editions.append({
        "ville": ville,
        "annee": annee,
        "titre": titre.strip(),
        "graveur": graveur_ou_inconnu(row.get(COL_GRAVEUR, "")),
        "publisher": row.get("publisher", "").strip() or "Éditeur non identifié",
        "lien": extraire_lien(row),
        "copies": row.get("copies de cette édition", "").strip(),
    })

# "s.n." ("sine nomine" : éditeur non mentionné sur l'édition) n'est pas un nom d'éditeur —
# plusieurs éditions "s.n." dans une même ville ne sont pas forcément du même éditeur. Sans
# ce correctif, elles seraient regroupées à tort en un seul point sur la carte (comme si
# c'était un unique éditeur très actif). On les distingue donc par un numéro (s.n.1, s.n.2...),
# uniquement quand il y en a plus d'une dans la ville (sinon le numéro n'apporte rien).
for ville in ORDRE_VILLES:
    inconnus = [e for e in editions if e["ville"] == ville and e["publisher"].strip().lower() == "s.n."]
    if len(inconnus) > 1:
        for i, e in enumerate(sorted(inconnus, key=lambda e: e["annee"]), start=1):
            e["publisher"] = f"s.n.{i}"

print(len(editions), "éditions retenues (Lyon, Paris, Venise)")
for v in ORDRE_VILLES:
    sous = [e for e in editions if e["ville"] == v]
    print(f"  {v:8s} {len(sous):2d} éditions, {min(e['annee'] for e in sous)}–{max(e['annee'] for e in sous)}, "
          f"{len({e['publisher'] for e in sous})} éditeurs distincts, "
          f"{len({e['graveur'] for e in sous})} graveurs distincts")

4 lignes fusionnées (tomes d'une même édition regroupés)
62 éditions retenues (Lyon, Paris, Venise)
  Lyon     19 éditions, 1510–1697, 14 éditeurs distincts, 8 graveurs distincts
  Paris    25 éditions, 1493–1737, 19 éditeurs distincts, 13 graveurs distincts
  Venise   18 éditions, 1497–1624, 14 éditeurs distincts, 10 graveurs distincts


## 2. Choix de visualisation


In [3]:
import math

# Mêmes coordonnées que 01_carte_circulation.ipynb
VILLES_COORDS = {"Lyon": (45.764, 4.8357), "Paris": (48.8566, 2.3522), "Venise": (45.4408, 12.3155)}

RAYON_MAX_KM = 22            # étalement maximal du nuage d'éditeurs autour de chaque ville
ANGLE_OR = math.radians(137.508)  # angle d'or : répartition régulière en spirale, sans grille

def decalage_spirale(i, n, rayon_max_km):
    """Spirale de phyllotaxie (rayon ∝ √i, angle = i × angle d'or) : répartit n'importe quel
    nombre de points régulièrement autour d'un centre, sans qu'ils se chevauchent."""
    if n <= 1:
        return 0.0, 0.0
    rayon = rayon_max_km * math.sqrt(i / (n - 1))
    angle = i * ANGLE_OR
    return rayon * math.cos(angle), rayon * math.sin(angle)

def deplacer_latlon(lat, lon, dx_km, dy_km):
    """Déplace un point de (dx_km vers l'est, dy_km vers le nord) en degrés lat/lon."""
    dlat = dy_km / 111.0
    dlon = dx_km / (111.0 * math.cos(math.radians(lat)))
    return lat + dlat, lon + dlon

# Une entrée par édition, pour le tableau détaillé de chaque ville (section 3).
points = [
    {
        "ville": e["ville"],
        "annee": e["annee"],
        "titre": e["titre"],
        "graveur": e["graveur"],
        "editeur": e["publisher"],
        "lien": e["lien"],
    }
    for e in editions
]

print(len(points), "éditions prêtes pour les tableaux détaillés")

62 éditions prêtes pour les tableaux détaillés


In [4]:
import colorsys

def couleur_categorielle(i, saturation=0.55, luminosite=0.48):
    """Couleur hex distincte pour l'index i, par rotation à l'angle d'or (voir ANGLE_OR
    plus haut) : n'importe quel nombre de catégories reste bien réparti sur le cercle
    chromatique, sans avoir à choisir une palette manuelle à l'avance."""
    teinte = (i * 137.508 % 360) / 360
    r, g, b = colorsys.hls_to_rgb(teinte, luminosite, saturation)
    return "#{:02x}{:02x}{:02x}".format(round(r * 255), round(g * 255), round(b * 255))

groupes_editeurs = {}
for e in editions:
    groupes_editeurs.setdefault((e["ville"], e["publisher"]), []).append(e)

points_editeurs = []
for ville in VILLES_COORDS:
    editeurs_ville = sorted(
        {cle[1] for cle in groupes_editeurs if cle[0] == ville},
        key=lambda editeur: (-len(groupes_editeurs[(ville, editeur)]), editeur)
    )
    n = len(editeurs_ville)
    lat0, lon0 = VILLES_COORDS[ville]
    for i, editeur in enumerate(editeurs_ville):
        couleur = couleur_categorielle(i)
        eds = sorted(groupes_editeurs[(ville, editeur)], key=lambda e: e["annee"])
        dx, dy = decalage_spirale(i, n, RAYON_MAX_KM)
        lat, lon = deplacer_latlon(lat0, lon0, dx, dy)
        points_editeurs.append({
            "lat": round(lat, 5),
            "lon": round(lon, 5),
            "ville": ville,
            "editeur": editeur,
            "couleur": couleur,
            "editions": [
                {"annee": e["annee"], "titre": e["titre"], "graveur": e["graveur"], "lien": e["lien"]}
                for e in eds
            ],
        })

print(len(points_editeurs), "éditeurs positionnés sur la carte (taille ∝ nombre d'éditions, couleur propre à chacun)")
for ville in VILLES_COORDS:
    sous = [p for p in points_editeurs if p["ville"] == ville]
    plus_actif = max(sous, key=lambda p: len(p["editions"]))
    print(f"  {ville:8s} {len(sous):2d} éditeurs — le plus actif : {plus_actif['editeur']} "
          f"({len(plus_actif['editions'])} éditions)")

47 éditeurs positionnés sur la carte (taille ∝ nombre d'éditions, couleur propre à chacun)
  Lyon     14 éditeurs — le plus actif : Jean de Tournes (5 éditions)
  Paris    19 éditeurs — le plus actif : Henri de Marnef et Guillaume Cavellat (5 éditions)
  Venise   14 éditeurs — le plus actif : Francesco de' Franceschi (4 éditions)


In [5]:
# Même logique que points_editeurs, mais groupée par graveur plutôt que par éditeur.
groupes_graveurs = {}
for e in editions:
    groupes_graveurs.setdefault((e["ville"], e["graveur"]), []).append(e)

points_graveurs = []
for ville in VILLES_COORDS:
    graveurs_ville = sorted(
        {cle[1] for cle in groupes_graveurs if cle[0] == ville},
        key=lambda graveur: (-len(groupes_graveurs[(ville, graveur)]), graveur)
    )
    n = len(graveurs_ville)
    lat0, lon0 = VILLES_COORDS[ville]
    for i, graveur in enumerate(graveurs_ville):
        couleur = couleur_categorielle(i)
        eds = sorted(groupes_graveurs[(ville, graveur)], key=lambda e: e["annee"])
        dx, dy = decalage_spirale(i, n, RAYON_MAX_KM)
        lat, lon = deplacer_latlon(lat0, lon0, dx, dy)
        points_graveurs.append({
            "lat": round(lat, 5),
            "lon": round(lon, 5),
            "ville": ville,
            "graveur": graveur,
            "couleur": couleur,
            "editions": [
                {"annee": e["annee"], "titre": e["titre"], "editeur": e["publisher"], "lien": e["lien"]}
                for e in eds
            ],
        })

print(len(points_graveurs), "graveurs positionnés sur la carte (taille ∝ nombre d'éditions, couleur propre à chacun)")
for ville in VILLES_COORDS:
    sous = [p for p in points_graveurs if p["ville"] == ville]
    plus_actif = max(sous, key=lambda p: len(p["editions"]))
    print(f"  {ville:8s} {len(sous):2d} graveurs — le plus actif : {plus_actif['graveur']} "
          f"({len(plus_actif['editions'])} éditions)")

31 graveurs positionnés sur la carte (taille ∝ nombre d'éditions, couleur propre à chacun)
  Lyon      8 graveurs — le plus actif : Leroy II, Guillaume (5 éditions)
  Paris    13 graveurs — le plus actif : Salomon, Bernard (5 éditions)
  Venise   10 graveurs — le plus actif : Anonyme1572 (5 éditions)


## 2bis. Liens entre points (même graveur / copie)

Deux types de liens, dessinés **au sein d'une même carte de ville uniquement** (jamais entre
deux villes : les trois cartes restent indépendantes, chacune cadrée sur son propre nuage —
contrairement à `01_carte_circulation.ipynb`, où tout est sur une seule carte). Une relation
qui relierait deux villes différentes (un même graveur actif à la fois à Lyon et à Paris, par
exemple) n'est donc pas représentée ici.

- **Vue Éditeur** : trait plein entre deux éditeurs de la même ville ayant chacun publié une
  édition d'un même graveur identifié ; trait pointillé (tirets longs) quand un éditeur publie
  une copie de la série d'un graveur déjà publié par un autre éditeur de la même ville.
- **Vue Graveur** : trait pointillé (fin) entre deux graveurs de la même ville quand l'un
  copie la série de l'autre.

Les copies sont détectées à partir de la colonne `copies de cette édition`, avec la même
logique qu'en `01_carte_circulation.ipynb` (`extraire_candidats_copie`, `trouver_graveur_connu`
— dupliquée à l'identique ici, comme `fusionner_tomes` l'est déjà dans les autres notebooks) :
mentions ambiguës ("copie X ou Y", "X ? Y ?") écartées sans lien plutôt que de deviner.
Différence avec `01` : le registre des instances par graveur est construit uniquement à partir
des éditions de Lyon/Paris/Venise (pas du corpus entier), puisque seules les instances de la
**même ville** que l'édition copieuse peuvent de toute façon donner lieu à un lien ici.

In [6]:
# --- Vue Éditeur : trait plein "même graveur" ---
# Deux éditeurs de la même ville ayant chacun publié au moins une édition d'un même graveur
# identifié ("Graveur non identifié" ne compte pas comme un point commun : ça regrouperait à
# tort des éditeurs sans lien réel connu). Un même couple d'éditeurs peut partager plusieurs
# graveurs : un seul lien est tracé, avec la liste complète en infobulle.
partages_graveur = {}
for e in editions:
    if e["graveur"] == "Graveur non identifié":
        continue
    partages_graveur.setdefault((e["ville"], e["graveur"]), set()).add(e["publisher"])

liens_editeurs_meme_graveur = {}
for (ville, graveur), eds in partages_graveur.items():
    eds = sorted(eds)
    for i in range(len(eds)):
        for j in range(i + 1, len(eds)):
            cle = (ville, eds[i], eds[j])
            liens_editeurs_meme_graveur.setdefault(cle, set()).add(graveur)

liens_editeurs_meme_graveur = [
    {"ville": ville, "a": a, "b": b, "graveurs": sorted(graveurs)}
    for (ville, a, b), graveurs in liens_editeurs_meme_graveur.items()
]
print(len(liens_editeurs_meme_graveur), "liens 'même graveur' entre éditeurs (vue Éditeur)")

30 liens 'même graveur' entre éditeurs (vue Éditeur)


In [7]:
# --- Traits "copie" (vues Éditeur et Graveur) ---
# Même logique de détection que 01_carte_circulation.ipynb (colonne "copies de cette édition"),
# dupliquée à l'identique (voir note de la section 2bis) — seule différence : le registre des
# instances par graveur ne couvre que Lyon/Paris/Venise, puisqu'un lien inter-villes n'est de
# toute façon pas tracé ici.

def normaliser_graveur(nom):
    """Retire les dates entre parenthèses et la ponctuation superflue."""
    nom = re.sub(r"\(.*?\)", "", nom)
    nom = nom.strip().rstrip(",").strip()
    return re.sub(r"\s+", " ", nom)

def normaliser_candidat_anonyme(c):
    """« Anonyme 1563 » ou un simple « 1572 » -> 'AnonymeXXXX' (même convention que le registre)."""
    c = re.sub(r"anonyme\s*(\d{4})", r"Anonyme\1", c, flags=re.IGNORECASE)
    if re.fullmatch(r"\d{4}", c.strip()):
        c = "Anonyme" + c.strip()
    return c.strip()

def eclater_enumeration(fragment):
    """« Anonyme1563, 1572 » -> ['Anonyme1563', '1572'] si tout ressemble à une liste d'années."""
    parties = [p.strip() for p in fragment.split(",")]
    if len(parties) > 1 and all(re.fullmatch(r"(anonyme\s*)?\d{4}", p, re.IGNORECASE) for p in parties):
        return parties
    return [fragment]

def mention_ambigue(texte):
    """"ou" ou "?" signale une attribution hésitante entre plusieurs graveurs : on ne devine pas."""
    return bool(re.search(r"\bou\b", texte, re.IGNORECASE)) or "?" in texte

def extraire_candidats_copie(texte):
    """« copie X et Y » / « même famille que X, Y et Z » -> liste de noms de graveurs candidats.
    (Les mentions ambiguës "ou"/"?" ont déjà été écartées par mention_ambigue avant cet appel.)"""
    t = re.sub(r"\(.*?\)", "", texte)
    t = re.sub(r"^\s*(copie|même famille que)\s*", "", t, flags=re.IGNORECASE)
    bruts = re.split(r"\s+et\s+", t)
    candidats = []
    for c in bruts:
        c = c.strip(" .,;?\xa0")
        if not c:
            continue
        for sous in eclater_enumeration(c):
            sous = normaliser_candidat_anonyme(sous.strip(" .,;?\xa0"))
            if sous:
                candidats.append(sous)
    return candidats

def tokens_nom(nom):
    nom = re.sub(r"\(.*?\)", "", nom)
    nom = nom.replace(",", " ")
    return set(t.lower() for t in re.findall(r"[a-zà-öø-ÿ']+", nom) if len(t) >= 3)

def trouver_graveur_connu(candidat, registre):
    """Fait correspondre un nom en texte libre à une clé du registre (chevauchement de mots)."""
    if candidat in registre:
        return candidat
    tc = tokens_nom(candidat)
    if not tc:
        return None
    meilleur, meilleur_score = None, 0
    for nom_reg in registre:
        if nom_reg.lower().startswith("anonyme"):
            continue  # déjà couvert par la correspondance exacte ci-dessus
        score = len(tc & tokens_nom(nom_reg))
        if score > meilleur_score:
            meilleur, meilleur_score = nom_reg, score
    return meilleur if meilleur_score >= 1 else None

# Registre : graveur normalisé -> instances (ville, année, éditeur), restreint à Lyon/Paris/Venise.
toutes_instances_par_graveur = {}
for e in editions:
    if e["graveur"] == "Graveur non identifié":
        continue
    g = normaliser_graveur(e["graveur"])
    toutes_instances_par_graveur.setdefault(g, []).append((e["ville"], e["annee"], e["publisher"]))

liens_editeurs_copie = []
liens_graveurs_copie = []
non_resolus = []

for e in editions:
    texte = e["copies"]
    if not texte:
        continue
    if mention_ambigue(texte):
        non_resolus.append((e["ville"], e["annee"], e["titre"], texte, "attribution ambiguë (ou/possibilité multiple)"))
        continue

    matches_vus = set()
    for candidat in extraire_candidats_copie(texte):
        match = trouver_graveur_connu(candidat, toutes_instances_par_graveur)
        if match is None:
            non_resolus.append((e["ville"], e["annee"], e["titre"], texte, f"aucune correspondance ({candidat!r})"))
            continue
        if match in matches_vus:
            continue
        matches_vus.add(match)

        # Seules les instances de la MÊME ville, antérieures ou égales, comptent ici (lien
        # intra-ville uniquement — cf. section 2bis).
        instances = [
            inst for inst in toutes_instances_par_graveur[match]
            if inst[0] == e["ville"] and inst[1] <= e["annee"] and inst != (e["ville"], e["annee"], e["publisher"])
        ]
        if not instances:
            non_resolus.append((e["ville"], e["annee"], e["titre"], texte, f"pas d'édition antérieure de {match!r} dans la même ville"))
            continue
        _, an_orig, editeur_orig = max(instances, key=lambda x: x[1])

        if editeur_orig != e["publisher"]:
            liens_editeurs_copie.append({
                "ville": e["ville"], "a": editeur_orig, "b": e["publisher"],
                "graveur": match, "titre_copie": e["titre"], "an_orig": an_orig, "an_dest": e["annee"],
            })
        graveur_dest = normaliser_graveur(e["graveur"])
        if match != graveur_dest:
            liens_graveurs_copie.append({
                "ville": e["ville"], "a": match, "b": graveur_dest,
                "titre_copie": e["titre"], "an_orig": an_orig, "an_dest": e["annee"],
            })

print(len(liens_editeurs_copie), "liens de copie intra-ville entre éditeurs (vue Éditeur)")
print(len(liens_graveurs_copie), "liens de copie intra-ville entre graveurs (vue Graveur)")
print(len(non_resolus), "mentions non résolues ou hors-scope (copie inter-villes, ambiguë...) :")
for ville, an, titre, texte, raison in non_resolus:
    print(f"  ✗ {ville} {an} « {titre[:50]} » — {texte!r} ({raison})")

4 liens de copie intra-ville entre éditeurs (vue Éditeur)
6 liens de copie intra-ville entre graveurs (vue Graveur)
14 mentions non résolues ou hors-scope (copie inter-villes, ambiguë...) :
  ✗ Lyon 1510 « P. Ovidii Nasonis Metamorphoseos libri moralizati » — 'copie Anonyme1497' (pas d'édition antérieure de 'Anonyme1497' dans la même ville)
  ✗ Lyon 1527 « Publii Ovidii Nasonis Sulmonensis Metamorphoseos L » — 'copie Leroy II, Guillaume ou 1497' (attribution ambiguë (ou/possibilité multiple))
  ✗ Lyon 1532 « Le Grand Olympe » — 'copie Leroy II, Guillaume ou Anonyme1497 ou Anonyme1527' (attribution ambiguë (ou/possibilité multiple))
  ✗ Lyon 1556 « Trois Premiers livres de la Métamorphose d'Ovide » — 'copie Bernard Salomon' (pas d'édition antérieure de 'Salomon, Bernard' dans la même ville)
  ✗ Paris 1606 « Les Metamorphoses d´Ovide de nouveau traduites en  » — 'copie Giacomo Franco' (pas d'édition antérieure de 'Franco, Giacomo' dans la même ville)
  ✗ Paris 1610 « Les Metamorphoses d´

## 3. Génération de la carte (HTML autonome)

In [8]:
def ligne_tableau(p):
    lien_html = f'<a href="{p["lien"]}" target="_blank">voir</a>' if p["lien"] else ""
    return (
        f'<tr><td>{p["annee"]}</td><td>{p["titre"]}</td>'
        f'<td>{p["editeur"]}</td><td>{p["graveur"]}</td>'
        f'<td>{lien_html}</td></tr>'
    )

def tableau_ville(ville):
    lignes = "\n".join(
        ligne_tableau(p) for p in sorted(points, key=lambda p: p["annee"]) if p["ville"] == ville
    )
    return f'''<table class="tableau-detaille" id="tableau-{ville}">
    <thead><tr><th>Année</th><th>Titre</th><th>Éditeur</th><th>Graveur</th><th>Lien</th></tr></thead>
    <tbody>
      {lignes}
    </tbody>
  </table>'''

# Une section par ville : carte, bouton pour déplier le tableau détaillé, puis le tableau.
cellules = []
for ville in VILLES_COORDS:
    cellules.append(f'''<div class="cellule-carte">
    <h2>{ville}</h2>
    <div class="carte" id="carte-{ville}"></div>
    <button class="bascule action-ville" data-ville="{ville}">Afficher le tableau détaillé</button>
    {tableau_ville(ville)}
  </div>''')
blocs_carte = "\n".join(cellules)

TEMPLATE_HTML = r"""<!DOCTYPE html>
<html lang="fr"><head><meta charset="utf-8">
<title>Nuage de points : éditions de Venise, Paris et Lyon</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<link href="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.css" rel="stylesheet"/>
<script src="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.js"></script>
<script src="https://unpkg.com/@maplibre/maplibre-gl-leaflet@0.0.20/leaflet-maplibre-gl.js"></script>
<style>
  :root {
    --surface: #fffaf0; --texte-fort: #2b1e15; --texte-att: #6b5c4f; --trait: #d8cfc0;
    --contour-point: #3e2c23; --c-lien: #2a78d6; --c-lien-meme-graveur: #8a7a63; --c-lien-copie: #a1502f;
  }
  @media (prefers-color-scheme: dark) {
    :root {
      --surface: #1a1a19; --texte-fort: #f2ece2; --texte-att: #c3baa9; --trait: #3a352c;
      --contour-point: #f2ece2; --c-lien: #3987e5; --c-lien-meme-graveur: #a99a80; --c-lien-copie: #c96b45;
    }
  }
  body { margin:0; font-family:Georgia,serif; background:var(--surface); color:var(--texte-fort); }
  .page { max-width:1100px; margin:0 auto; padding:16px 20px 32px; position:relative; }
  h1 { font-size:19px; margin:0 0 4px; }
  p.souschapo { font-size:13px; color:var(--texte-att); margin:0 0 10px; }

  .commandes-globales { display:flex; align-items:center; gap:8px; font-size:12px;
    color:var(--texte-att); margin:0 0 6px; }
  .bascule-groupement { font-family:Georgia,serif; font-size:12px; background:none;
    border:1px solid var(--trait); color:var(--texte-fort); border-radius:4px; padding:5px 10px;
    cursor:pointer; }
  .bascule-groupement.actif { background:var(--texte-fort); color:var(--surface); border-color:var(--texte-fort); }

  /* Légende des traits entre points : son contenu change selon le regroupement (éditeur/graveur). */
  .legende-liens { display:flex; flex-wrap:wrap; align-items:center; gap:14px; font-size:11.5px;
    color:var(--texte-att); margin:0 0 6px; }
  .legende-liens .item { display:flex; align-items:center; gap:5px; }
  .legende-liens .trait { display:inline-block; width:26px; height:0; border-top-width:2px; border-top-style:solid; }
  .legende-liens .trait.meme-graveur { border-color:var(--c-lien-meme-graveur); border-top-style:solid; }
  .legende-liens .trait.copie-editeur { border-color:var(--c-lien-copie); border-top-style:dashed; }
  .legende-liens .trait.copie-graveur { border-color:var(--c-lien-copie); border-top-style:dotted; border-top-width:2.5px; }
  .note-survol { font-size:11.5px; color:var(--texte-att); font-style:italic; margin:0 0 14px; }

  /* Les 3 cartes empilées verticalement, une par ville. */
  .rangee-cartes { display:flex; flex-direction:column; gap:20px; margin-top:14px; }
  .cellule-carte h2 { font-size:15px; margin:0 0 6px; color:var(--texte-fort); }
  .carte { height:380px; border-radius:6px; box-shadow:0 1px 6px rgba(0,0,0,.25); }

  .action-ville { display:block; margin:8px 0 0; }
  button.bascule { font-family:Georgia,serif; font-size:12px; background:none;
    border:1px solid var(--trait); color:var(--texte-fort); border-radius:4px; padding:5px 10px;
    cursor:pointer; }
  table.tableau-detaille { width:100%; border-collapse:collapse;
    font-size:12px; margin:6px 0 0; display:none; }
  table.tableau-detaille.visible { display:table; }
  table.tableau-detaille th, table.tableau-detaille td { text-align:left; padding:4px 8px;
    border-bottom:1px solid var(--trait); }
  table.tableau-detaille a { color:var(--c-lien); }

  .leaflet-tooltip { font-family:Georgia,serif; font-size:12px; background:var(--surface);
    border:1px solid var(--contour-point); color:var(--texte-fort); padding:4px 9px;
    box-shadow:0 1px 5px rgba(0,0,0,.3); }
  /* Popups Leaflet (clic sur un éditeur ou un graveur) : même thème parchemin, contenu
     défilable si la liste d'éditions est longue, liens "voir" bien cliquables (contrairement
     à une infobulle au survol, une popup reste ouverte tant qu'on ne clique pas ailleurs). */
  .leaflet-popup-content-wrapper { background:var(--surface); color:var(--texte-fort);
    border:1px solid var(--contour-point); border-radius:6px; }
  .leaflet-popup-tip { background:var(--surface); border:1px solid var(--contour-point); }
  .leaflet-popup-content { font-family:Georgia,serif; font-size:12px; max-height:220px;
    overflow-y:auto; margin:10px 12px; }
  .leaflet-popup-content hr { border:none; border-top:1px solid var(--trait); margin:4px 0; }
  .ed-popup { padding:3px 0; border-bottom:1px solid var(--trait); }
  .ed-popup:last-child { border-bottom:none; }
  .ed-popup a { color:var(--c-lien); }
</style></head><body>
<div class="page">
  <h1>Éditions de Venise, Paris et Lyon</h1>
  <p class="souschapo">Chaque point regroupe les éditions d'un même éditeur (ou graveur) : sa
    taille indique combien. Survolez pour un aperçu, cliquez pour le détail de chaque édition
    et son lien vers la version numérisée.</p>
  <div class="commandes-globales">
    <span>Regrouper les points de la carte par :</span>
    <button class="bascule-groupement actif" data-mode="editeur">Éditeur</button>
    <button class="bascule-groupement" data-mode="graveur">Graveur</button>
  </div>
  <div class="legende-liens" id="legendeLiens"></div>
  <p class="note-survol">Les traits relient des points d'une même ville (jamais entre deux
    villes). Survolez un trait pour afficher son détail.</p>
  <div class="rangee-cartes">
    __BLOCS_CARTE__
  </div>
</div>
<script>
  const pointsEditeurs = __POINTS_EDITEURS__;
  const pointsGraveurs = __POINTS_GRAVEURS__;
  const villesCoords = __VILLES_COORDS__;
  const liensEditeursMemeGraveur = __LIENS_EDITEURS_MEME_GRAVEUR__;
  const liensEditeursCopie = __LIENS_EDITEURS_COPIE__;
  const liensGraveursCopie = __LIENS_GRAVEURS_COPIE__;

  const LEGENDES_MODE = {
    editeur: '<div class="item"><span class="trait meme-graveur"></span>même graveur publié par les deux éditeurs</div>'
      + '<div class="item"><span class="trait copie-editeur"></span>un éditeur copie la série d\'un graveur déjà publié par l\'autre</div>',
    graveur: '<div class="item"><span class="trait copie-graveur"></span>un graveur copie la série de l\'autre</div>',
  };

  // Styles des traits, affichés en permanence (pas d'état "repos/actif" : simple et direct).
  // Chaque trait est accompagné d'un halo clair dessous (cf. construireLiens), pour qu'il se
  // détache nettement du fond de carte historique (déjà chargé en toponymes/routes/rivières).
  const STYLE_MEME_GRAVEUR = {color:'#8a7a63', dashArray:null, weight:2, opacity:.75, weightHalo:5, opaciteHalo:.4};
  const STYLE_COPIE_EDITEUR = {color:'#a1502f', dashArray:'9,6', weight:2.2, opacity:.85, weightHalo:5.2, opaciteHalo:.4};
  const STYLE_COPIE_GRAVEUR = {color:'#a1502f', dashArray:'1,7', lineCap:'round', weight:2.4, opacity:.85, weightHalo:5.4, opaciteHalo:.4};
  const COULEUR_HALO = '#fffbe8';

  // Construit les marqueurs Leaflet d'un jeu de points (éditeurs ou graveurs) pour une ville.
  // `cle` vaut 'editeur' ou 'graveur' : nom de la clé qui porte le nom du groupe dans `p`.
  // Chaque édition listée dans la popup affiche en plus l'autre role (le graveur quand les
  // points sont des éditeurs, l'éditeur quand les points sont des graveurs) : le nom du
  // groupe lui-même est déjà dans l'entête de la popup, mais pas cette autre information,
  // qui elle varie d'une édition à l'autre au sein d'un même point.
  function construireMarqueurs(map, liste, ville, cle) {
    const autreCle = cle === 'editeur' ? 'graveur' : 'editeur';
    return liste.filter(p => p.ville === ville).map(p => {
      const n = p.editions.length;
      const rayon = 6 + Math.sqrt(n) * 5;
      const nom = p[cle];
      const editionsTriees = p.editions.slice().sort((a, b) => a.annee - b.annee);
      const listeEditions = editionsTriees.map(e =>
        '<div class="ed-popup"><b>' + e.annee + '</b> — ' + e.titre +
        ' <i>(' + e[autreCle] + ')</i>' +
        (e.lien ? ' <a href="' + e.lien + '" target="_blank">→ voir</a>' : '') + '</div>'
      ).join('');
      return L.circleMarker([p.lat, p.lon], {
        radius: rayon, color:'#3e2c23', weight:1, fillColor: p.couleur, fillOpacity:.85
      })
        // survol : aperçu rapide (nom + nombre d'éditions), pas de lien
        .bindTooltip('<b>' + nom + '</b><br>' + ville + ' · ' + n + ' édition(s)', {sticky:true, maxWidth:220})
        // clic : popup Leaflet, nativement interactive, avec le détail de chaque édition et
        // son lien "voir"
        .bindPopup('<b>' + nom + '</b><br>' + ville + ', ' + n + ' édition(s)<hr>' + listeEditions, {maxWidth:280});
    });
  }

  // Index de position (ville|nom -> {lat, lon}) pour placer les traits entre deux points déjà
  // positionnés par la spirale, sans recalculer quoi que ce soit.
  function construireIndex(liste, cle) {
    const index = {};
    liste.forEach(p => { index[p.ville + '|' + p[cle]] = p; });
    return index;
  }
  const indexEditeurs = construireIndex(pointsEditeurs, 'editeur');
  const indexGraveurs = construireIndex(pointsGraveurs, 'graveur');

  // Construit les traits d'un type de lien pour une ville : chaque trait est une paire
  // {halo, trait} (halo clair dessous, trait coloré dessus), toujours affichée au même niveau
  // d'opacité — pas d'état "repos/actif" ni de survol de point qui les ferait apparaître ou
  // disparaître. Seul le survol du trait lui-même affiche son détail (bindTooltip). Un lien
  // dont un des deux points n'existe pas dans l'index (ne devrait pas arriver, les deux venant
  // du même calcul côté notebook) est silencieusement ignoré plutôt que de planter l'affichage.
  function construireLiens(liens, index, ville, style, infobulle) {
    return liens.filter(l => l.ville === ville).flatMap(l => {
      const pA = index[ville + '|' + l.a], pB = index[ville + '|' + l.b];
      if (!pA || !pB) return [];
      const pts = [[pA.lat, pA.lon], [pB.lat, pB.lon]];
      const halo = L.polyline(pts, {
        color: COULEUR_HALO, weight: style.weightHalo, opacity: style.opaciteHalo, interactive: false,
      });
      const trait = L.polyline(pts, {
        color: style.color, weight: style.weight, opacity: style.opacity,
        dashArray: style.dashArray, lineCap: style.lineCap || 'round',
      }).bindTooltip(infobulle(l), {sticky:true, maxWidth:240});
      return [halo, trait];
    });
  }

  // Une carte Leaflet par ville, chacune cadrée sur son propre nuage (pas sur les 3 villes à
  // la fois : Lyon/Paris/Venise sont loin les unes des autres, une seule carte les englobant
  // montrerait surtout du vide entre elles). Fond OpenHistoricalMap (frontières et toponymes
  // d'époque), comme dans 01_carte_circulation.ipynb. Les deux jeux de marqueurs (éditeur et
  // graveur), chacun avec ses traits, sont construits à l'avance dans des L.layerGroup séparés :
  // basculer ne fait que retirer l'un et ajouter l'autre, sans rien reconstruire. Les traits
  // sont ajoutés avant les marqueurs dans chaque layerGroup pour que les points restent
  // visuellement au-dessus des lignes qui les relient.
  const calquesParVille = {};
  for (const ville of Object.keys(villesCoords)) {
    const map = L.map('carte-' + ville, {scrollWheelZoom:false});
    L.maplibreGL({
      style: 'https://www.openhistoricalmap.org/map-styles/main/main.json',
      attribution: '© OpenHistoricalMap contributors'
    }).addTo(map);

    const editeursVille = pointsEditeurs.filter(p => p.ville === ville);
    const bornes = L.latLngBounds(editeursVille.map(p => [p.lat, p.lon]));
    map.fitBounds(bornes, {padding:[36, 36]});

    const traitsEditeur = [
      ...construireLiens(liensEditeursMemeGraveur, indexEditeurs, ville, STYLE_MEME_GRAVEUR,
        l => l.a + ' & ' + l.b + ' — même graveur : ' + l.graveurs.join(', ')),
      ...construireLiens(liensEditeursCopie, indexEditeurs, ville, STYLE_COPIE_EDITEUR,
        l => l.b + ' (' + l.an_dest + ') copie la série de ' + l.graveur + ', déjà publiée par ' + l.a + ' (' + l.an_orig + ')'),
    ];
    const traitsGraveur = construireLiens(liensGraveursCopie, indexGraveurs, ville, STYLE_COPIE_GRAVEUR,
      l => l.b + ' copie ' + l.a + ' — « ' + l.titre_copie + ' » (' + l.an_dest + ')');

    const calqueEditeur = L.layerGroup([...traitsEditeur, ...construireMarqueurs(map, pointsEditeurs, ville, 'editeur')]);
    const calqueGraveur = L.layerGroup([...traitsGraveur, ...construireMarqueurs(map, pointsGraveurs, ville, 'graveur')]);
    calqueEditeur.addTo(map);
    calquesParVille[ville] = {map, editeur: calqueEditeur, graveur: calqueGraveur};
  }

  // Bascule globale "Éditeur" / "Graveur" : change le calque affiché sur les 3 cartes, et la
  // légende des traits (les deux vues n'ont pas le même sens de trait plein/pointillé).
  let modeGroupement = 'editeur';
  document.getElementById('legendeLiens').innerHTML = LEGENDES_MODE[modeGroupement];
  document.querySelectorAll('.bascule-groupement').forEach(bouton => {
    bouton.addEventListener('click', () => {
      const mode = bouton.dataset.mode;
      if (mode === modeGroupement) return;
      document.querySelectorAll('.bascule-groupement').forEach(b => b.classList.toggle('actif', b === bouton));
      for (const ville of Object.keys(calquesParVille)) {
        const c = calquesParVille[ville];
        c.map.removeLayer(c[modeGroupement]);
        c.map.addLayer(c[mode]);
      }
      modeGroupement = mode;
      document.getElementById('legendeLiens').innerHTML = LEGENDES_MODE[modeGroupement];
    });
  });

  // Un bouton "Afficher le tableau détaillé" par ville, juste sous sa carte.
  document.querySelectorAll('.action-ville').forEach(bouton => {
    bouton.addEventListener('click', () => {
      const tableau = document.getElementById('tableau-' + bouton.dataset.ville);
      const visible = tableau.classList.toggle('visible');
      bouton.textContent = visible ? 'Masquer le tableau détaillé' : 'Afficher le tableau détaillé';
    });
  });
</script>
</body></html>"""

html_final = (TEMPLATE_HTML
    .replace("__BLOCS_CARTE__", blocs_carte)
    .replace("__POINTS_EDITEURS__", json.dumps(points_editeurs, ensure_ascii=False))
    .replace("__POINTS_GRAVEURS__", json.dumps(points_graveurs, ensure_ascii=False))
    .replace("__VILLES_COORDS__", json.dumps(VILLES_COORDS, ensure_ascii=False))
    .replace("__LIENS_EDITEURS_MEME_GRAVEUR__", json.dumps(liens_editeurs_meme_graveur, ensure_ascii=False))
    .replace("__LIENS_EDITEURS_COPIE__", json.dumps(liens_editeurs_copie, ensure_ascii=False))
    .replace("__LIENS_GRAVEURS_COPIE__", json.dumps(liens_graveurs_copie, ensure_ascii=False)))

with open(CHEMIN_SORTIE, "w", encoding="utf-8") as f:
    f.write(html_final)

print("Carte écrite dans", CHEMIN_SORTIE)

Carte écrite dans /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/resultats/Datavis/nuage_editions_venise_paris_lyon.html
